# Lagrangian Trajectory Modelling in the Gulf of Guinea with OpenDrift
### Oil spills (2D & 3D), Search-and-Rescue drift, and marine plastic transport
!["lagrangian schematic"](figures/lagrangian_modelling_schematic_GGOSSS2026.png)

# GGOSSS Summer School — Physical Oceanography & Operational Forecasting Track

**Instructors:** 
- Fernand Assene     — NASMOST, University of Ebolowa, Kribi, Cameroon — fernandassene@hotmail.com
- Ibrahima sidiki BA — LPAOF/ESP, Université Cheikh Anta Diop de Dakar — ibrahimasidiki.ba@ucad.edu.sn
- Hugo Plombat       — University of Michigan                          — plombat@umich.edu 

---

## Why this notebook exists

The Gulf of Guinea (GoG) is one of the busiest and most vulnerable maritime regions in the world:

- **Oil & gas traffic**: tanker routes serving Nigerian, Cameroonian, Gabonese and Equatorial Guinean terminals (Bonny, Kribi–Lolabé, Port-Gentil, Malabo) cross a narrow, energetic coastal current system.
- **Piracy and SAR incidents**: the GoG accounts for a large share of global reported piracy/kidnapping incidents at sea (ICC IMB, 2023), making person-in-water and vessel-drift forecasting operationally important.
- **Plastic pollution**: high coastal population density (Lagos, Douala, Abidjan, Accra) combined with river discharge (Niger, Sanaga, Ogooué, Congo) makes the GoG a regional plastic accumulation zone.

This notebook builds, step by step, **three operational Lagrangian applications** using **[OpenDrift](https://opendrift.github.io/)** (Dagestad et al., 2018), forced first by **coarse global reanalysis (CMEMS GLORYS12, 1/12°)** and then by a **regional, coastal-resolving CROCO simulation (1/36°)**, so that you can *see with your own eyes* why resolution matters for coastal, boundary-current-dominated regions like ours.

## Learning objectives

By the end of this notebook you will be able to:

1. Set up and configure OpenDrift's `OpenOil`, `Leeway`, and `OceanDrift`/plastic modules.
2. Force a Lagrangian model with CMEMS GLORYS12 reanalysis and with regional CROCO output.
3. Run 2D (surface) and 3D (with vertical processes) oil-spill simulations.
4. Run a Search-And-Rescue (SAR) drift simulation for a person-in-water and a disabled vessel.
5. Run a multi-source (river-input) plastic-transport simulation representative of GoG river inputs.
6. **Critically compare** GLORYS12 (1/12°, ~9 km) vs CROCO (1/36°, ~3 km) forcing and explain, quantitatively, the operational consequences of under-resolving coastal eddies, jets and shelf currents.

## Structure

| Part | Content |
|---|---|
| 0 | Environment setup |
| 1 | Study area & forcing data (GLORYS12 vs CROCO) |
| 2 | Oil spill — 2D surface (OpenOil) |
| 3 | Oil spill — 3D with weathering & vertical mixing |
| 4 | Search and Rescue: person-in-water & vessel (Leeway) |
| 5 | Marine plastic transport (multi-source, seasonal) |
| 6 | **Resolution exercise**: GLORYS12 vs CROCO, guided student exercises |
| 7 | Going operational: CROCO forecasts + free atmospheric forecasts (CMEMS, GFS, ECMWF open data, WRF) |
| 8 | References |

> **Convention used throughout:** all input files are read from the `data/` folder and all outputs (NetCDF
> trajectories, figures) are written to `outputs/`. File names, domain, dates and release points are set **once**,
> in the configuration cell of Part 0. The exercises are stated at the end of each Part; worked answers are in
> the companion `OpenDrift_GulfOfGuinea_SummerSchool_SOLUTIONS.ipynb`.

## Part 0 — Environment setup

OpenDrift is pure Python and depends on `netCDF4`, `xarray`, `cartopy`, `pyproj` and `scipy`. We also use
`copernicusmarine` (the official CMEMS client, replacing the deprecated `motuclient`/`cmems` toolboxes since 2023)
to download GLORYS12 data.

**Reference versions used to write/check this notebook:** OpenDrift ≥ 1.11, copernicusmarine ≥ 2.0, xarray ≥ 2024.x.
Always check `opendrift.__version__` — the API for readers and configuration changed between major versions.


Small, reusable diagnostics (convex-hull area, stranded fraction, centroid distance, coastal sectors, EKE, ...)
live in `gog_drift_utils.py` next to this notebook, so that this notebook and the solutions notebook compute
them in exactly the same way. Open it: it is short, commented, plain numpy/xarray code.

In [ ]:
# Run once per environment (comment out if already installed)
# !pip install opendrift copernicusmarine xarray netCDF4 cartopy cmocean cfgrib bottleneck --quiet

import opendrift
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cmocean
from pathlib import Path
from datetime import datetime, timedelta

import gog_drift_utils as gu   # shared helper functions (same folder as this notebook)

print("OpenDrift version:", opendrift.__version__)
print("xarray version   :", xr.__version__)

### 0.1 Configuration — the only cell you need to edit

> **About the sample files currently in `data/`.** They are *placeholders* used to check that every cell runs:
> they do **not** cover the Gulf of Guinea scenario below (the CROCO and MERCATOR files cover the Canary
> region at different dates; the ERA5 wind covers the Gulf of Guinea, May–Sep 2026). With these files the
> coverage check in Part 1.5 will warn you, and OpenDrift will use fallback values (e.g. zero current) where
> no data exist: trajectories are then *not meaningful*. Once GoG files for the same zone and period are
> placed in `data/`, update the file names and dates below — nothing else in the notebook needs to change.

In [ ]:
# ============================== CONFIGURATION (edit here only) ==============================
DATA_DIR = Path("data")        # all input files
OUT_DIR  = Path("outputs")     # all outputs (trajectory NetCDF files, figures)
OUT_DIR.mkdir(exist_ok=True)

DOWNLOAD_DATA   = False   # True -> (re)download forcing with copernicusmarine / cdsapi / GFS (needs accounts + internet)
MAKE_ANIMATIONS = False   # True -> also write .mp4 animations (slow; needs ffmpeg)

# --- Domain and period --------------------------------------------------------------------
lon_min, lon_max = 6.0, 10.0
lat_min, lat_max = 1.0, 5.0
start = datetime(2023, 8, 1)
end   = datetime(2023, 8, 10)    # 10-day window: enough for a spill/SAR case study

# --- Forcing files (hindcast) ---------------------------------------------------------------
glorys_file = DATA_DIR / "MERCATOR_20260918_00.nc"   # GLORYS12-type product on a regular lon/lat grid (daily means)
croco_file  = DATA_DIR / "croco_his.nc"              # CROCO history file (native s-coordinates, C-grid)
era5_wind_components = [DATA_DIR / "ARCO_u10_20260501_20260922.nc",   # ERA5 10 m wind: u10 and v10 may be
                        DATA_DIR / "ARCO_v10_20260501_20260922.nc"]   # in one file or in two separate files
wind_file   = DATA_DIR / "era5_wind_gog.nc"          # merged, CF-compliant wind file built in Part 1.4

# --- Forcing files (operational forecast, Part 7) --------------------------------------------
glorys_fc_file = DATA_DIR / "glorys_forecast_gog.nc"                   # CMEMS global analysis & forecast
croco_fc_file  = croco_file                                            # today's CROCO forecast cycle (placeholder)
gfs_file       = DATA_DIR / "gfs_wind_gog_forecast.nc"                 # GFS 10 m wind forecast (Part 7.4)
wrf_wind_file  = DATA_DIR / "wrf_3km_gog_wind_forecast_regridded.nc"   # optional regional WRF winds (Part 7.6)

# --- Scenarios --------------------------------------------------------------------------------
release_lon, release_lat = 9.42, 2.78          # oil spill: Kribi-Lolabe terminal approach, approx.
release_time = start + timedelta(hours=2)
oil_mass_kg  = 300_000                          # 300 t instantaneous surface release

last_known_position = (9.55, 3.05)              # SAR: lon, lat of the reported man-overboard position
incident_time = start + timedelta(hours=6)
search_delay  = timedelta(hours=3)              # realistic time-to-launch of a regional coast-guard asset

river_mouths = {                                 # plastic sources: lon, lat of river mouths
    "Sanaga": (9.73, 3.58),
    "Nyong":  (9.87, 3.25),
}

## Part 1 — Study area and forcing data

### 1.1 Domain

We use a Gulf of Guinea sub-domain centred on the Cameroonian coast, covering the Kribi–Douala–Bight of Bonny
shelf: **longitude 6°E–10°E, latitude 1°N–5°N**. This box contains:

- the Kribi–Lolabé deep-water oil terminal and approach channel,
- the Sanaga and Nyong river plumes (freshwater/plastic sources),
- part of the seasonal Guinea Current / coastal upwelling system described for the region by
  Djakouré et al. (2017) and Da-Allada et al. (2017).

### 1.2 Ocean forcing #1 — CMEMS GLORYS12 reanalysis (1/12°, ~9 km)

GLORYS12V1 (Lellouche et al., 2021) is the CMEMS global ocean eddy-resolving (at mid-latitudes) reanalysis,
1/12° horizontal resolution, 50 vertical levels, assimilating altimetry, SST and in-situ profiles. At equatorial
latitudes such as ours (1 grid cell ≈ 9–11 km near the equator), it is only marginally eddy-*permitting* and
does **not** resolve the internal Rossby radius of deformation, which in the Gulf of Guinea is of the order of
20–40 km but with narrow coastal jets and fronts much smaller than that.

Data are retrieved from the CMEMS Marine Data Store product
`GLOBAL_MULTIYEAR_PHY_001_030` (dataset id `cmems_mod_glo_phy_my_0.083deg_P1D-m`).

In [ ]:
if DOWNLOAD_DATA:
    import copernicusmarine
    # --- Credentials: set once, stored securely by the client (do not hardcode passwords in shared notebooks) ---
    # copernicusmarine.login()   # run interactively once; stores a token in ~/.copernicusmarine
    copernicusmarine.subset(
        dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
        variables=["uo", "vo", "so", "thetao", "zos"],   # u, v, salinity, temperature, sea-surface height
        minimum_longitude=lon_min, maximum_longitude=lon_max,
        minimum_latitude=lat_min,  maximum_latitude=lat_max,
        start_datetime=start.isoformat(), end_datetime=end.isoformat(),
        minimum_depth=0, maximum_depth=100,   # keep upper 100 m: enough for surface + subsurface oil/plastic
        output_directory=str(DATA_DIR), output_filename=glorys_file.name,
    )
    print("Saved:", glorys_file)
else:
    print("DOWNLOAD_DATA is False -> using the existing file:", glorys_file, "(exists:", glorys_file.exists(), ")")

### 1.3 Ocean forcing #2 — CROCO 1/36° regional simulation (~3 km, coastal-resolving)

For the same period and domain, we use a nested **CROCO** (Coastal and Regional Ocean COmmunity model;
Auclair et al., 2000; Debreu et al., 2012 for AGRIF nesting) simulation at **1/36°** horizontal resolution
(~3 km at the equator), forced at its open boundaries by GLORYS12 and at the surface by ERA5 atmospheric
forcing. At 3 km resolution the model explicitly resolves mesoscale-to-submesoscale coastal eddies, the Sanaga/Nyong river
plume fronts, and the narrow along-shelf jet — none of which are resolved at 1/12°.

We assume the CROCO history file (`croco_file` in the configuration cell, in `data/`) has already been produced (e.g. from your own run) and follows the standard CROCO/ROMS terrain-following (s-coordinate) output convention with
`u`, `v` on the Arakawa C-grid and a vertical stretching defined by `theta_s`, `theta_b`, `hc`.

In [ ]:
ds_croco = xr.open_dataset(croco_file)
print(ds_croco[["u", "v", "zeta", "temp"]])

# Horizontal grid spacing: a degree of longitude is 111.32*cos(latitude) km
dlon = float(ds_croco.lon_rho.diff("xi_rho").mean())
dlat = float(ds_croco.lat_rho.diff("eta_rho").mean())
lat_mid = float(ds_croco.lat_rho.mean())
print(f"\nCROCO grid spacing ~ {dlon:.4f} deg lon x {dlat:.4f} deg lat"
      f"  ->  ~{dlon*111.32*np.cos(np.deg2rad(lat_mid)):.2f} km x {dlat*111.32:.2f} km (at {lat_mid:.1f} deg N)")

> **Note on CROCO readers in OpenDrift:** OpenDrift reads native CROCO/ROMS output through
> `opendrift.readers.reader_ROMS_native`, which correctly handles the Arakawa C-grid staggering and the
> s-coordinate vertical transform (it needs `hc`, `Cs_r`/`Cs_w`, `theta_s`, `theta_b`, and either `Vtransform`/`Vstretching`
> or older `sc_r` variables present in the file — check with `ncdump -h` if the reader fails to find them).
> GLORYS12, being a regular lon/lat NetCDF product, is instead read through the generic
> `opendrift.readers.reader_netCDF_CF_generic`, which auto-detects CF-compliant coordinate/variable names.

In [ ]:
from opendrift.readers import reader_netCDF_CF_generic, reader_ROMS_native

reader_glorys = reader_netCDF_CF_generic.Reader(str(glorys_file))
# CROCO history files often lack the u/v land masks required by the reader: gu.croco_gridfile() derives them
# from mask_rho into a small side file passed as `gridfile` (returns None if the masks are already present).
croco_masks = gu.croco_gridfile(croco_file, DATA_DIR / f"{croco_file.stem}_uv_masks.nc")
reader_croco  = reader_ROMS_native.Reader(str(croco_file), gridfile=croco_masks and str(croco_masks))

print(reader_glorys)
print(reader_croco)

### 1.4 Wind forcing

Both oil and Leeway (SAR) modules need a wind field (direct windage on oil slicks/floating objects). We use
ERA5 hourly 10 m winds (Hersbach et al., 2020), retrieved for the same domain/period via the Copernicus
Climate Data Store (`cdsapi`), and add it as an additional reader so OpenDrift can combine currents + wind.


> ERA5 10 m winds are sometimes delivered as **one file per component** (`u10`, `v10`). They must be *merged* as
> two variables of one dataset, not concatenated in time (`cdo cat u.nc v.nc` silently keeps only `u10` — the
> cause of the *"Readers must be added for ... y_wind"* error). `gu.prepare_wind_file` merges the component
> files, renames them to the CF names OpenDrift expects (`x_wind`, `y_wind`) and writes `data/era5_wind_gog.nc`.

In [ ]:
if DOWNLOAD_DATA:
    import cdsapi
    c = cdsapi.Client()
    c.retrieve('reanalysis-era5-single-levels', {
        'product_type': 'reanalysis',
        'variable': ['10m_u_component_of_wind', '10m_v_component_of_wind'],
        'area': [lat_max + 1, lon_min - 1, lat_min - 1, lon_max + 1],   # N, W, S, E (+1 deg margin)
        'date': f"{start:%Y-%m-%d}/{end:%Y-%m-%d}",
        'time': [f"{h:02d}:00" for h in range(24)],
        'data_format': 'netcdf',
    }, str(DATA_DIR / "era5_wind_raw.nc"))
    era5_wind_components = [DATA_DIR / "era5_wind_raw.nc"]

gu.prepare_wind_file(era5_wind_components, wind_file)   # does nothing if wind_file already exists
reader_wind = reader_netCDF_CF_generic.Reader(str(wind_file))
print(reader_wind)

### 1.5 Coverage check

Before running anything, check that every reader covers the release point and the simulation period. Where
no reader covers a particle, OpenDrift silently uses *fallback* values (zero current, for instance): the run
completes, but the trajectories are meaningless.

In [ ]:
gu.check_coverage([reader_glorys, reader_croco, reader_wind],
                  release_lon, release_lat, release_time, release_time + timedelta(hours=72));

## Part 2 — Oil spill: 2D surface simulation (`OpenOil`)

### 2.1 Scenario

A **medium crude oil release near the Kribi–Lolabé terminal approach**, similar in spirit to historical
incidents in the region (e.g. the 2021 *Trinity Spirit* FPSO explosion off Nigeria). We model a **300-tonne
surface release** of a Nigerian light crude (`OSEII` category used by NOAA ADIOS database as analogue) at the
terminal approach point.

We first run the model **purely at the surface (2D)**: no vertical entrainment, oil stays at `z=0`. This is
the fast, first-look configuration typically run within the first hour of an incident notification.

In [ ]:
from opendrift.models.openoil import OpenOil

o2d = OpenOil(loglevel=20)  # 20 = INFO; use 0 for DEBUG when troubleshooting

o2d.add_reader([reader_glorys, reader_wind])

# --- Oil type: choose closest analogue from the NOAA ADIOS oil library bundled with OpenDrift ---
o2d.set_config('seed:oil_type', 'GENERIC MEDIUM CRUDE')

# --- 2D configuration: disable vertical entrainment/mixing, force oil to stay at surface ---
o2d.set_config('processes:evaporation', True)
o2d.set_config('processes:emulsification', True)
o2d.set_config('processes:dispersion', False)     # no wave-induced natural dispersion in this 2D pass
o2d.set_config('drift:vertical_mixing', False)    # <-- the key "2D" switch
o2d.set_config('drift:current_uncertainty', 0.1)  # m/s, stochastic sub-grid spread
o2d.set_config('drift:wind_uncertainty', 1.0)     # m/s
o2d.set_config('general:coastline_action', 'stranding')   # default: elements hitting the coast are deactivated

# NB: OpenOil.seed_elements() ignores a `mass_oil=` argument (the mass per element is recomputed from
# `m3_per_hour` and the oil density). gu.seed_oil_mass() seeds, then rescales the per-element mass so that
# the TOTAL released mass is exactly oil_mass_kg.
gu.seed_oil_mass(
    o2d, oil_mass_kg,
    lon=release_lon, lat=release_lat,
    radius=100,                       # m, initial spill patch radius
    number=2000,                      # Lagrangian elements (particles)
    time=release_time,
    z=0,
)

o2d.run(duration=timedelta(hours=72), time_step=900, time_step_output=3600,
        outfile=str(OUT_DIR / "oil_2D_glorys_kribi.nc"))

In [ ]:
o2d.plot(fast=True, filename=str(OUT_DIR / "oil_2D_glorys_kribi_map.png"))
if MAKE_ANIMATIONS:
    o2d.animation(filename=str(OUT_DIR / "oil_2D_glorys_kribi.mp4"), fast=True)  # optional, slower

# Mass balance is the standard "first question" from any incident commander:
o2d.plot_oil_budget(filename=str(OUT_DIR / "oil_2D_glorys_kribi_budget.png"))
budget = o2d.get_oil_budget()
print("Oil budget after 72 h (% of released mass):")
for k in ['mass_surface', 'mass_submerged', 'mass_stranded', 'mass_evaporated', 'mass_dispersed']:
    print(f"  {k:16s}: {100 * float(budget[k][-1] / budget['mass_total'][0]):5.1f} %")

### 2.2 Reading the mass balance

`OpenOil` tracks oil mass in several budget categories: `mass_oil` (remaining liquid oil, per element), `mass_evaporated`,
`mass_dispersed` (into the water column, if enabled) and `mass_biodegraded` (long runs only). `o.get_oil_budget()`
sums them over all elements and splits the remaining oil into *surface*, *submerged* and *stranded*. For an
incident briefing you should always be able to answer: *how much oil is still floating, in three days?*

---

### EXERCISE 2.1 — Oil-type sensitivity

Re-run the same oil-spill scenario using `GENERIC LIGHT CRUDE` and `GENERIC HEAVY CRUDE`.
Compare the three simulations over the first 72 hours.

1. Plot the evaporation curves for the three oil types.
2. Quantify the amount of oil remaining after 72 hours for each type.
3. Based on these results alone, describe what differs between the three simulations (do not yet explain why).
4. Check the API gravity of the three oil records. Which analogue is numerically closest to the typical API
   gravity of Nigerian Bonny Light crude (~35–37° API)? *(After seeding, the selected record is `o.oiltype`;
   `gu.oil_properties(o)` lists its API gravity and other properties.)*
5. Now, propose a physical explanation for the differences observed in (2), (3), and discuss why API gravity
   alone may not be sufficient to fully characterize an oil for a spill simulation (consider at least one other
   property available in the oil record).

### EXERCISE 2.2 — Effect of the release position

Move the release location approximately 15 km offshore while keeping the other simulation settings unchanged
(`gu.offshore_point` can choose the offshore direction for you).

1. Display the original and modified release locations on a map.
2. Run the simulation for 72 hours for both locations.
3. Quantify the fraction of particles reaching the coastline in each case (elements with `status = 'stranded'`),
   and compare the two particle distributions.
4. Discuss how the initial distance from the coastline may influence the simulated coastal impact, and identify
   at least one other factor (besides distance) that could also explain part of the difference you observe.

## Part 3 — Oil spill: 3D simulation with weathering and vertical mixing

Real oil spills are **not** 2D: wave breaking entrains oil droplets into the water column (natural dispersion),
droplets of different size resurface at different rates (Stokes' law), and near-shore turbulent mixing can
push oil metres below the surface, changing predicted landfall location and timing substantially.

OpenOil implements 3D droplet dynamics following Johansen (2000) and Röhrs et al. (2018): a droplet-size
distribution is generated at the sea surface (function of oil viscosity, interfacial tension and wave energy),
and each droplet's terminal rise velocity is computed from Stokes'/empirical drag laws, then advected in 3D by
`u, v, w` currents plus turbulent vertical mixing.

In [ ]:
o3d = OpenOil(loglevel=20)
o3d.add_reader([reader_glorys, reader_wind])

o3d.set_config('seed:oil_type', 'GENERIC MEDIUM CRUDE')
o3d.set_config('processes:evaporation', True)
o3d.set_config('processes:emulsification', True)
o3d.set_config('processes:dispersion', True)          # wave-induced natural dispersion -> ON
o3d.set_config('drift:vertical_mixing', True)          # <-- the key "3D" switch
o3d.set_config('vertical_mixing:diffusivitymodel', 'windspeed_Large1994')
# Vertical ADVECTION by a resolved w is left OFF: GLORYS12 provides no vertical velocity (OpenDrift would use
# w = 0 anyway). This 3D configuration therefore only activates turbulent vertical mixing + droplet rise
# (see Exercise 3.2, question 5, for a run with an explicit w from CROCO).
o3d.set_config('drift:vertical_advection', False)
o3d.set_config('drift:current_uncertainty', 0.1)
o3d.set_config('drift:wind_uncertainty', 1.0)
o3d.set_config('vertical_mixing:timestep', 60)         # s, finer sub-stepping needed for the mixing scheme

gu.seed_oil_mass(
    o3d, oil_mass_kg,
    lon=release_lon, lat=release_lat,
    radius=100, number=2000,
    time=release_time, z=0,
)

o3d.run(duration=timedelta(hours=72), time_step=900, time_step_output=3600,
        outfile=str(OUT_DIR / "oil_3D_glorys_kribi.nc"))

In [ ]:
# Depth of the elements through time: median and 5-95 % range of z (0 = surface)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, o, title in [(axes[0], o2d, "2D run: no vertical mixing"),
                     (axes[1], o3d, "3D run: with entrainment + mixing")]:
    z = o.result.z
    hours = (o.result.time - o.result.time[0]) / np.timedelta64(1, 'h')
    ax.fill_between(hours, z.quantile(0.05, dim='trajectory'), z.quantile(0.95, dim='trajectory'),
                    alpha=0.3, label='5-95 % of elements')
    ax.plot(hours, z.median(dim='trajectory'), label='median')
    ax.plot(hours, z.min(dim='trajectory'), 'k:', lw=0.8, label='deepest element')
    ax.set_title(title); ax.set_xlabel("Hours since release"); ax.grid(alpha=0.3)
axes[0].set_ylabel("z [m]"); axes[1].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "oil_2D_vs_3D_depth.png", dpi=150)
plt.show()

### 3.1 Why 3D matters operationally in the Gulf of Guinea

- **Shear dispersion**: currents typically rotate/change speed with depth in the coastal boundary layer;
  oil mixed 5–10 m down can travel in a *different direction* than surface-only oil, changing which coastline
  is impacted.
- **Resurfacing timing**: for the Niger Delta / Bight of Bonny shallow shelf, small droplets can stay
  submerged long enough to be advected past a response window before resurfacing further along the coast.
- **Booming/skimming decisions**: responders need a **3D-informed submerged-oil mass fraction**, not just a
  2D surface footprint, to decide whether subsurface dispersant application is warranted.

---

### EXERCISE 3.1 — Effect of wind conditions on oil dispersion

Select one relatively low-wind period and one relatively high-wind period from the available atmospheric forcing.
Run the same oil-spill scenario (3D configuration) for both periods.

1. Quantify and compare the wind conditions during the two simulations (mean and maximum wind speed).
2. Quantify the amount of oil dispersed into the water column after 48 hours in each case.
3. Is the change in dispersed fraction proportional to the change in wind speed? Show this with numbers before
   answering yes or no.
4. Discuss which physical processes could explain a non-proportional response, if you find one.

### EXERCISE 3.2 — Two-dimensional versus three-dimensional transport

Compare the 72-hour oil-spill simulations performed with two-dimensional and three-dimensional ocean forcing
(`o2d` and `o3d` above).

1. Compare the spatial distribution of particles at the surface, and the extent/shape of the simulated surface footprint.
2. For the three-dimensional simulation only, examine the vertical distribution of particles.
3. Based on (1), (2), describe, without yet explaining, how the two simulations differ.
4. Discuss how vertical mixing and depth-dependent currents can modify the transport pathway, and why a
   surface-only simulation may differ from a three-dimensional one.
5. *(Going further, optional)* The 3D configuration used here only activates turbulent vertical mixing. If you
   have access to a 3D current field with an explicit vertical velocity `w`, repeat the comparison with
   `drift:vertical_advection` enabled and discuss whether resolved upwelling/downwelling changes your conclusion in (4).

## Part 4 — Search and Rescue: person-in-water and vessel drift (`Leeway`)

### 4.1 Background

The `Leeway` model (Breivik et al., 2011) predicts the drift of a floating object (person, life raft, disabled
vessel) as a combination of:

$$\mathbf{v}_{leeway} = \mathbf{v}_{current} + \alpha\,\mathbf{v}_{wind,10m} + \mathbf{v}_{jibe/uncertainty}$$

where α (the "leeway coefficient") and a downwind-angle offset are empirically calibrated **per object category**
(over 60 categories are built into OpenDrift, from "person-in-water, vertical PFD" to "fishing vessel, drifting").
This is the model used operationally by maritime SAR authorities worldwide (incl. IMRF / Cospas-Sarsat guidance).

Given the frequency of piracy-related kidnappings and small-craft incidents reported in the GoG (ICC IMB
annual piracy reports), a locally-relevant SAR exercise is directly useful for coast guard and civil-protection
partners of the GGOSSS network.

In [ ]:
from opendrift.models.leeway import Leeway

sar = Leeway(loglevel=20)
sar.add_reader([reader_croco, reader_wind])   # NOTE: using the higher-resolution CROCO reader here on purpose

# sar.leewayprop is a dict {integer id: properties}; the object category is selected by its exact
# 'Description' (or 'OBJKEY') through the 'seed:object_type' configuration setting.
leeway_table = pd.DataFrame(sar.leewayprop).T
print(f"{len(leeway_table)} object categories available, e.g.:")
print(leeway_table.loc[leeway_table.Description.str.contains('PIW|Person|Fishing vessel'),
                       ['OBJKEY', 'Description', 'DWSLOPE', 'DWOFFSET', 'CWRSLOPE', 'CWLSLOPE']].to_string())

person_category = "Person-in-water (PIW), unknown state (mean values)"
vessel_category = "Fishing vessel, general (mean values)"

sar.set_config('seed:object_type', person_category)
sar.seed_elements(
    lon=last_known_position[0], lat=last_known_position[1],
    time=incident_time,
    number=1000,
    radius=500,           # m, position uncertainty of last known fix
)

sar.run(duration=timedelta(hours=48), time_step=600, time_step_output=1800,
        outfile=str(OUT_DIR / "sar_person_croco.nc"))

In [ ]:
sar.plot(fast=True, filename=str(OUT_DIR / "sar_person_croco_map.png"))

# Standard SAR product: where are the particles when the SAR asset is launched (incident + search_delay)?
# Here: particle density on a ~1 km grid at that time, from this single run.
hours_launch = search_delay / timedelta(hours=1)
st = gu.state_at(sar.result, hours_launch)
ext = gu.extent_of(sar.result, pad=0.3)
fig = plt.figure(figsize=(7, 6))
ax = gu.map_axes(fig, extent=ext, title=f"PIW particle density at launch time (T+{hours_launch:.0f} h)")
H, xe, ye = np.histogram2d(st.lon.dropna(), st.lat.dropna(), bins=[np.arange(ext[0], ext[1], 0.01),
                                                                    np.arange(ext[2], ext[3], 0.01)])
import cartopy.crs as ccrs
pm = ax.pcolormesh(xe, ye, np.ma.masked_equal(H.T, 0), cmap="cmo.amp", transform=ccrs.PlateCarree(), zorder=3)
plt.colorbar(pm, ax=ax, shrink=0.7, label="particles per 0.01 deg cell")
ax.plot(*last_known_position, 'k*', ms=12, transform=ccrs.PlateCarree(), zorder=4, label="last known position")
ax.legend(loc="lower left")
plt.savefig(OUT_DIR / f"sar_person_search_area_T+{hours_launch:.0f}h.png", dpi=150)
plt.show()

### 4.2 Vessel drift comparison

Re-run for a **drifting fishing vessel** (different windage/leeway angle than a person), and compare search
areas — vessels typically show a stronger, more consistent downwind offset than a person-in-water.

In [ ]:
sar_vessel = Leeway(loglevel=20)
sar_vessel.add_reader([reader_croco, reader_wind])

sar_vessel.set_config('seed:object_type', vessel_category)
sar_vessel.seed_elements(
    lon=last_known_position[0], lat=last_known_position[1],
    time=incident_time, number=1000, radius=500,
)
sar_vessel.run(duration=timedelta(hours=48), time_step=600, time_step_output=1800,
               outfile=str(OUT_DIR / "sar_vessel_croco.nc"))

sar_vessel.plot(fast=True, filename=str(OUT_DIR / "sar_vessel_croco_map.png"))

### EXERCISE 4.1 — Person-in-water versus drifting vessel

Compare the simulated trajectories of a person-in-water and a drifting vessel under the same environmental forcing.

1. Plot the trajectories or ensemble distributions for the two object types.
2. Compare their final positions after 24 hours, and calculate the separation distance between the two ensemble centroids.
3. Examine the leeway parameters (windage, downwind/crosswind coefficients) used for the two object categories
   (`sar.leewayprop`, printed above).
4. Based on (2) and (3), explain how differences in windage and leeway can influence the simulated trajectories.

### EXERCISE 4.2 — Effect of ocean forcing on the simulated SAR footprint

Repeat the person-in-water simulation using GLORYS12 and CROCO ocean forcing. Compare the two simulations after 24 hours.

1. Plot the particle distributions for both ocean forcings.
2. Calculate the convex-hull area of each particle ensemble, and the relative difference between the two areas.
   *Note: the convex-hull area is a simple geometric envelope of the simulated particles at one instant; it is a
   convenient proxy, not an actual probabilistic search area. A real SAR search-area product is normally built
   from an ensemble of runs (varying the release point, forcing, or leeway category) and expressed as a
   probability density, not from a single deterministic run.*
3. Compare the shapes and spatial distributions of the two ensembles.
4. Discuss how differences in the simulated ocean circulation may influence the predicted search area, and what
   this implies for how confidently a single-run convex-hull area should be reported to a SAR team.

**Safety note:** These are **teaching exercises on synthetic/illustrative incidents**. Never use notebook
output as the sole basis for a real SAR decision — always couple with a certified operational system (e.g.
national MRCC tools, Cospas-Sarsat SARSAT products) and human expert judgement.

## Part 5 — Marine plastic transport

### 5.1 Scenario

We represent **river-sourced plastic input** from the Sanaga and Nyong rivers over a full seasonal cycle,
following the general river-plastic-flux approach of Lebreton et al. (2017) and Jambeck et al. (2015) (used
to rank West/Central African rivers among significant plastic sources), and track it with a `OceanDrift`
configuration tuned for buoyant, wind-affected floating debris (a simplified alternative to the dedicated
`opendrift.models.plastdrift` module, shown below for comparison).

In [ ]:
from opendrift.models.oceandrift import OceanDrift

plastic = OceanDrift(loglevel=20)
plastic.add_reader([reader_glorys, reader_wind])

plastic.set_config('drift:advection_scheme', 'runge-kutta4')
plastic.set_config('drift:vertical_mixing', False)   # floating plastic assumed to stay near-surface
plastic.set_config('drift:current_uncertainty', 0.05)
plastic.set_config('drift:wind_uncertainty', 0.5)
plastic.set_config('general:coastline_action', 'stranding')

seed_time_start = start
seed_time_end   = start + timedelta(days=9)
n_per_river = 500

for i, (river, (rlon, rlat)) in enumerate(river_mouths.items()):
    if gu.is_land(rlon, rlat)[0]:
        print(f"note: the {river} source ({rlon}, {rlat}) is on land in the GSHHG coastline -> OpenDrift moves "
              "the seeded elements to the nearest ocean position (seed:ocean_only = True)")
    plastic.seed_elements(
        lon=rlon, lat=rlat, radius=1000, number=n_per_river,
        time=[seed_time_start, seed_time_end],   # continuous release over the run window
        wind_drift_factor=0.02,                  # ~2% windage (an element property, set at seeding)
        origin_marker=i,
    )

plastic.run(duration=timedelta(days=20), time_step=1800, time_step_output=3600*6,
            outfile=str(OUT_DIR / "plastic_gog_glorys.nc"))

In [ ]:
plastic.plot(fast=True, linecolor='origin_marker', filename=str(OUT_DIR / "plastic_transport_map.png"))

# Stranding statistics per source river.
# OpenDrift writes NaN (also for 'status') after an element is deactivated, so the status at the last output
# time is NOT enough: gu.state_at() forward-fills each element's last valid status/position. The numeric
# 'stranded' code is read from the file metadata (flag_meanings) rather than assumed.
ds_plastic = xr.open_dataset(OUT_DIR / "plastic_gog_glorys.nc")
final = gu.state_at(ds_plastic)
stranded = gu.stranded_mask(ds_plastic)
for i, river in enumerate(river_mouths):
    from_river = (final.origin_marker.values == i)
    n = int((stranded & from_river).sum())
    print(f"{river}: {n} / {from_river.sum()} particles stranded within 20 days ({100*n/max(from_river.sum(),1):.1f} %)")

### 5.2 Beaching / coastline interaction

By default OpenDrift's landmask stops particles at the coastline (`status='stranded'`). This is a *numerical
treatment* of what happens when a particle's trajectory crosses the coastline, set via:

```python
plastic.set_config('general:coastline_action', 'stranding')  # default: element deactivated where it hits the coast
plastic.set_config('general:coastline_action', 'previous')   # element moved back to its last position in open water
```

Neither option is, by itself, a physical representation of beaching, adhesion or resuspension. `'previous'`
simply returns the particle to its last position in open water rather than marking it as stranded; it does not
simulate a physical bounce, current-driven resuspension, or any other real coastal process. Physically-based
beaching models (e.g. Onink et al., 2021) instead use probabilities of beaching/resuspension per time step.

---

### EXERCISE 5.1 — Interaction between particles and the coastline

Run the Sanaga plastic-release experiment using the two coastline treatments `coastline_action = 'stranding'`
and `coastline_action = 'previous'`.

1. Quantify the number and fraction of particles stranded after 20 days in each case.
2. Compare the spatial distribution of the particles for the two treatments.
3. Based on (1) and (2) only, describe what changes between the two runs.
4. Explain, in terms of what each option actually does numerically (see the note above), why this difference
   arises, and discuss what this means for interpreting "stranded fraction" as a real-world quantity.

### EXERCISE 5.2 — Effect of the source location on coastal plastic distribution

Add a third plastic-release source near the Wouri/Douala area (`lon=9.70, lat=4.05`) while keeping the other
model settings unchanged. Compare the three source locations.

1. Plot the final particle distributions for the three sources.
2. Quantify the fraction of particles reaching each of several predefined coastal sectors, for each source.
3. Identify which coastal sector receives the largest simulated fraction of particles, for each source.
4. Relate the differences you observe between source locations to the simulated surface circulation; is the
   relationship the same for all three sources, or does it depend on local conditions near each source?

## Part 6 — Why resolution matters: GLORYS12 (1/12°) vs CROCO (1/36°)

### 6.1 The physical argument

At equatorial latitudes, 1/12° (GLORYS12) corresponds to a grid spacing of **~9–10 km**, while 1/36° (CROCO)
corresponds to **~3 km**. Two physical scales in the Gulf of Guinea sit right at this boundary:

- The **first-mode baroclinic Rossby radius** in the tropical Atlantic is of order 30–80 km — GLORYS12
  marginally resolves it, but its associated **coastal jets, fronts and submesoscale eddies (< 20 km)**
  driving cross-shelf exchange are essentially filtered out by 1/12° resolution (Chelton et al., 2011, show
  this systematically for altimetry-class resolution; the same argument applies to models discretised at
  similar grid spacing).
- **River plume fronts** (Sanaga, Nyong, Wouri, Niger) have typical widths of 2–10 km — these are simply
  **not represented** at 1/12° and are the dominant near-field control on where oil or plastic released near
  a river mouth actually goes in the first hours.

CROCO's AGRIF one-way/two-way nesting (Debreu et al., 2012) allows a regional 1/36° domain to be forced by a
coarser parent (here GLORYS12) at its open boundaries while explicitly resolving these coastal-scale
processes internally.

### 6.2 Quantitative side-by-side comparison

We repeat the **2D oil spill scenario from Part 2**, changing only the ocean-current reader (same time step,
same number of elements, same wind), and compare: (i) the 72 h surface footprint, (ii) the time to first
coastal impact, (iii) its location.

> **Resolution is not the only difference.** GLORYS12 is used here as a *daily-mean* product, whereas the
> CROCO history file is typically written every few hours. Differences between the two runs can come from
> the spatial resolution, from the temporal sampling of the currents (tides, inertial oscillations and
> sea-breeze responses are averaged out of daily means), from the models' physics/forcing, or from the data
> assimilated in the reanalysis. Keep this in mind in Exercises 6.1–6.4. Print `reader.time_step` of each reader.

In [ ]:
o2d_croco = OpenOil(loglevel=20)
o2d_croco.add_reader([reader_croco, reader_wind])

o2d_croco.set_config('seed:oil_type', 'GENERIC MEDIUM CRUDE')
o2d_croco.set_config('processes:evaporation', True)
o2d_croco.set_config('processes:emulsification', True)
o2d_croco.set_config('processes:dispersion', False)
o2d_croco.set_config('drift:vertical_mixing', False)
o2d_croco.set_config('drift:current_uncertainty', 0.1)
o2d_croco.set_config('drift:wind_uncertainty', 1.0)
o2d_croco.set_config('general:coastline_action', 'stranding')

gu.seed_oil_mass(
    o2d_croco, oil_mass_kg,
    lon=release_lon, lat=release_lat, radius=100, number=2000,
    time=release_time, z=0,
)
# Same time step as the GLORYS12 run, so that only the ocean forcing differs between the two runs
o2d_croco.run(duration=timedelta(hours=72), time_step=900, time_step_output=3600,
              outfile=str(OUT_DIR / "oil_2D_croco_kribi.nc"))

print("Ocean forcing time step: GLORYS12 =", reader_glorys.time_step, "| CROCO =", reader_croco.time_step)

In [ ]:
ds_g = xr.open_dataset(OUT_DIR / "oil_2D_glorys_kribi.nc")
ds_c = xr.open_dataset(OUT_DIR / "oil_2D_croco_kribi.nc")

# OpenDrift's o.plot() always creates its own figure, so side-by-side panels are drawn with gu.plot_ensemble()
ext = gu.extent_of(ds_g, ds_c)
fig = plt.figure(figsize=(13, 6))
for k, (ds, title, col) in enumerate([(ds_g, "Forced by GLORYS12 (1/12°, ~9 km)", "C0"),
                                      (ds_c, "Forced by CROCO (1/36°, ~3 km)", "C3")]):
    ax = gu.map_axes(fig, extent=ext, subplot=(1, 2, k + 1), title=title)
    gu.plot_ensemble(ax, ds, color=col, hull=True)
plt.tight_layout()
plt.savefig(OUT_DIR / "resolution_comparison_oil.png", dpi=150)
plt.show()

for label, ds in [("GLORYS12", ds_g), ("CROCO   ", ds_c)]:
    fin = gu.state_at(ds)   # final positions (stranded elements kept at their stranding position)
    print(f"{label}-forced 72 h convex-hull footprint: {gu.hull_area_km2(fin.lon, fin.lat):8.1f} km^2")

### 6.3 Guided exercises — resolution impact

### EXERCISE 6.1 — Quantitative comparison of GLORYS12 and CROCO

Using the GLORYS12 and CROCO simulations, quantify the differences in particle transport. For each simulation, calculate:

1. the time to first coastal impact;
2. the fraction of particles reaching the coastline after 72 hours;
3. the final convex-hull area of the particle ensemble (see the note in Exercise 4.2: this remains a geometric
   envelope, not a physical footprint area);
4. the location of the first coastal impact.

Present the results in a table. Then:

5. Compare the trajectories and particle distributions obtained with the two ocean forcings and describe the
   differences first, without yet attributing them to a cause.
6. Discuss possible physical and numerical reasons for the differences between the two simulations. In
   particular, GLORYS12 is used here as a daily-mean product: before concluding that a given difference is due to
   spatial resolution alone, consider whether it could instead (or also) reflect the coarser temporal sampling of
   the current field.

### EXERCISE 6.2 — Surface eddy kinetic energy

Using the surface currents, calculate the time-mean surface eddy kinetic energy
$\mathrm{EKE} = \tfrac{1}{2}\left(\overline{u'^2 + v'^2}\right)$, where $u' = u - \bar u$ and $v' = v - \bar v$
are the anomalies of the zonal and meridional currents relative to their time-mean over the period considered.

> This is a decomposition against the temporal mean, not a spectral high-pass filter: no frequency cutoff is
> applied, so EKE computed this way includes all time scales of variability present in the record (mesoscale
> eddies, but also any seasonal or tidal signal that survives averaging over your chosen period). Keep this in
> mind when interpreting the results.

Calculate the EKE for both GLORYS12 and CROCO over the same region and period.

1. Plot the EKE fields for both products.
2. Compare their spatial structures and identify regions where EKE is relatively high in each product.
3. Based on (1) and (2) only, describe the differences between the two EKE fields.
4. Discuss possible reasons for these differences (grid resolution, ability to resolve mesoscale/submesoscale
   features, length of the averaging period, temporal sampling of the input currents).
5. Relate the observed EKE structures to the differences in particle transport found in Exercise 6.1: is a region
   of high EKE necessarily a region of large transport uncertainty in your simulations, or does that depend on
   where the release point sits relative to it?

### EXERCISE 6.3 — SAR briefing

Using the GLORYS12 and CROCO simulations, prepare a short briefing for a Search and Rescue team. Your briefing should include:

* the simulated 24-hour search area for each ocean forcing (stating clearly that this is a convex-hull envelope
  from a single deterministic run, not a probability-weighted search area);
* the percentage difference between the two areas;
* the main differences in the spatial distribution of the particle ensembles;
* possible physical or numerical explanations for the differences (see Exercise 6.1, question 6);
* at least one limitation that should be considered before using these simulations operationally (e.g. absence
  of an ensemble, daily-mean vs higher-frequency forcing, leeway category uncertainty).

Based on your analysis, discuss how the choice of ocean forcing can affect a SAR planning exercise, and what
additional information (beyond a single-run footprint) an operational team would normally want before
committing search assets.

### EXERCISE 6.4 — Plastic transport under different ocean forcings

Repeat the Wouri-source plastic simulation using CROCO forcing and compare the results with the GLORYS12 simulation,
in terms of:

1. final particle distribution;
2. fraction of stranded particles (keeping in mind the note in Exercise 5.1 on what "stranded" means numerically);
3. mean latitude of stranded particles;
4. spatial spread of the stranding locations.

Then:

5. Describe the main differences you observe between the two simulations.
6. Discuss possible physical reasons for these differences, considering the circulation represented by the two
   ocean models, and note whether any of the reasons you gave in Exercise 6.1 (question 6: resolution vs temporal
   sampling) also apply here.

> **Take-home message:** for *basin-scale* transport questions (weeks, hundreds of km), GLORYS12 is often
> sufficient and far cheaper to run. For **coastal impact questions on hours-to-days timescales** — exactly
> the oil-spill, SAR and near-field plastic questions that matter operationally in the Gulf of Guinea — a
> nested, coastal-resolving model such as CROCO at 1/36° is generally better suited, *provided* its added value
> is demonstrated (e.g. against drifters or HF radar) and the comparison accounts for temporal sampling and
> not only for grid spacing.

## Part 7 -- Going operational: CROCO *forecasts* and free atmospheric forecast products

Everything so far used **hindcast/reanalysis** forcing (GLORYS12 reanalysis, ERA5 reanalysis): perfect for
teaching and case-study reconstruction, but **useless for a real incident**, where you need a *forecast* of
where currents and wind will take the spill/vessel/debris over the next 24-120 h. This section shows how to
swap every reader used above for its **operational, freely available forecast equivalent**, with no change to
the OpenDrift model configuration itself -- only the readers change.

### 7.1 Two ways to force OpenDrift operationally

| | Ocean forcing | Atmospheric forcing |
|---|---|---|
| **Global, ready-to-use forecast** | CMEMS Global Analysis & Forecast (`GLOBAL_ANALYSISFORECAST_PHY_001_024`, 1/12 deg, updated daily, ~10-day forecast) | NOAA GFS (0.25 deg, 6-hourly, 16-day forecast, free via NOMADS) or ECMWF's free open-data forecast |
| **Regional, coastal-resolving forecast** | Your own **CROCO forecast run** (1/36 deg), boundary/initial-forced by the CMEMS global forecast above | Your own **WRF forecast run**, boundary-forced by GFS, or ECMWF's free open-data forecast |

The pedagogical point carried over from Part 6 applies with even more force here: an *operational* forecast
inherits the resolution limitations of its own forcing. A CROCO forecast is only as good as (a) its own 1/36 deg
internal dynamics **and** (b) the boundary/atmospheric forecast feeding it -- garbage/coarse boundaries in,
garbage/coarse forecast out, no matter how fine the internal CROCO grid is.

### 7.2 Ocean forecast option A -- CMEMS Global Analysis & Forecast (1/12 deg, no CROCO run needed)

For a quick operational response when no fresh regional CROCO forecast is available, CMEMS' **forecast**
product (as opposed to the **reanalysis** product used in Part 1) is retrieved exactly the same way, just a
different `dataset_id`, and is updated daily with a rolling ~10-day lead time.

In [ ]:
from datetime import timezone

# datetime.utcnow() is deprecated (Python >= 3.12); use timezone-aware UTC 'now', then drop tzinfo
# (OpenDrift and the NetCDF files use naive UTC datetimes).
now = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0, tzinfo=None)
forecast_start = now
forecast_end   = now + timedelta(days=5)   # 5-day operational window

if DOWNLOAD_DATA:
    import copernicusmarine
    copernicusmarine.subset(
        # Forecast (not reanalysis!) product: updated daily, rolling ~10-day horizon
        dataset_id="cmems_mod_glo_phy-cur_anfc_0.083deg_PT6H-i",
        variables=["uo", "vo"],
        minimum_longitude=lon_min, maximum_longitude=lon_max,
        minimum_latitude=lat_min,  maximum_latitude=lat_max,
        start_datetime=forecast_start.isoformat(), end_datetime=forecast_end.isoformat(),
        minimum_depth=0, maximum_depth=100,
        output_directory=str(DATA_DIR), output_filename=glorys_fc_file.name,
    )
    print("Saved operational forecast forcing:", glorys_fc_file)
else:
    print("DOWNLOAD_DATA is False -> expected forecast file:", glorys_fc_file, "(exists:", glorys_fc_file.exists(), ")")
# NOTE: dataset_id names on the Copernicus Marine Data Store are periodically renamed/reprocessed
# (e.g. "anfc" = analysis-and-forecast). Always confirm the current id at data.marine.copernicus.eu
# before an actual incident response, since product catalogues evolve over time.

### 7.3 Ocean forecast option B -- your own CROCO forecast run (1/36 deg, coastal-resolving)

A CROCO **forecast** run is structurally identical to the hindcast run used in Part 1/6, except:

1. Initial conditions and open-boundary conditions come from a **forecast** ocean product (e.g. the CMEMS
   `anfc` product above, or Mercator Ocean's global forecast system) instead of a reanalysis.
2. Surface forcing (wind, heat, freshwater fluxes) comes from an **atmospheric forecast** (Section 7.4/7.5
   below) instead of ERA5 reanalysis.
3. The run is launched on a rolling daily/twice-daily cycle, each time producing a short (typically 3-5 day)
   forecast history file.

Reading the resulting forecast history file into OpenDrift needs **no new code** -- it is still a CROCO
NetCDF history file, read with the same `reader_ROMS_native` used in Part 1/6:

In [ ]:
# Output of today's operational CROCO 1/36 GoG forecast cycle (produced by your own forecast pipeline);
# `croco_fc_file` is set in the configuration cell.
croco_fc_masks = gu.croco_gridfile(croco_fc_file, DATA_DIR / f"{croco_fc_file.stem}_uv_masks.nc")
reader_croco_fc = reader_ROMS_native.Reader(str(croco_fc_file), gridfile=croco_fc_masks and str(croco_fc_masks))
print(reader_croco_fc)

# From here on, every OpenDrift model (OpenOil / Leeway / OceanDrift) is set up EXACTLY as in Parts 2-5,
# simply passing `reader_croco_fc` (+ an atmospheric forecast reader, see 7.4-7.6) instead of the
# hindcast readers used earlier.

> **Practical tip:** keep a small `get_latest_forecast_reader()` helper function in your operational pipeline
> that always resolves to *this cycle's* file, so the exact same OpenDrift driver script used for teaching in
> Parts 2-5 can be reused unmodified for real-time response -- only the reader-construction step changes.

### 7.4 Atmospheric forecast option A -- NOAA GFS (free, global, 0.25 deg, via NOMADS)

The US NOAA **Global Forecast System (GFS)** is fully free, updated 4x/day (00/06/12/18 UTC), 0.25 deg
resolution, 16-day lead time, and is the de-facto free global standard used to force many operational
CROCO/WRF systems worldwide, including in West/Central Africa where a national ECMWF subscription may not be
available.


> **Access.** The NOMADS OPeNDAP (`/dods/`) service historically used to open GFS directly with xarray is no
> longer reliable. GFS is also published on the NOAA Open Data archive on AWS (`noaa-gfs-bdp-pds`), for the latest
> cycles *and* for archived cycles back to 2021. `gu.download_gfs_10m_wind` downloads only the 10 m wind
> records of each GRIB2 file (HTTP byte ranges, a few MB in total) and writes a CF NetCDF file for OpenDrift.

In [ ]:
# Most recent GFS cycle that is fully available (cycles appear ~4-5 h after their nominal time)
cycle = (now - timedelta(hours=6)).replace(hour=(now - timedelta(hours=6)).hour // 6 * 6)
lead_hours = range(0, 121, 3)   # 0-120 h, 3-hourly

if DOWNLOAD_DATA:
    gu.download_gfs_10m_wind(cycle, lead_hours, (lon_min - 1, lon_max + 1, lat_min - 1, lat_max + 1),
                             gfs_file, overwrite=True)
else:
    print("DOWNLOAD_DATA is False -> expected GFS file:", gfs_file, "(exists:", gfs_file.exists(), ")")

In [ ]:
reader_wind_gfs = reader_netCDF_CF_generic.Reader(str(gfs_file)) if gfs_file.exists() else None
print(reader_wind_gfs if reader_wind_gfs is not None else f"No GFS file ({gfs_file}) - set DOWNLOAD_DATA = True")

### 7.5 Atmospheric forecast option B -- ECMWF open-data forecasts (free, global, ~0.25 deg)

Since 2022, **ECMWF publishes a subset of its operational HRES/ENS forecasts for free**, under the
"ECMWF open data" programme (no CDS account/quota needed, unlike the full commercial/research archive),
via the `ecmwf-opendata` Python client. This is a good free alternative/cross-check to GFS, and can
outperform it in the tropics for some variables.

In [ ]:
if DOWNLOAD_DATA:
    try:
        from ecmwf.opendata import Client   # !pip install ecmwf-opendata
        client = Client(source="ecmwf")
        client.retrieve(
            date=0,           # 0 = today's most recent available cycle
            time=0,           # 00 UTC cycle
            step=[0, 24, 48, 72, 96, 120],
            stream="oper",
            type="fc",
            param=["10u", "10v"],
            target=str(DATA_DIR / "ecmwf_wind_forecast.grib2"),
        )
        # GRIB2 -> CF NetCDF readable by OpenDrift (global file: crop to the domain + 1 deg)
        ec = xr.open_dataset(DATA_DIR / "ecmwf_wind_forecast.grib2", engine="cfgrib").rename({"time": "cycle"})
        ec = ec.swap_dims({"step": "valid_time"}).rename({"valid_time": "time"}).drop_vars(["cycle", "step"], errors="ignore")
        ec.to_netcdf(DATA_DIR / "ecmwf_wind_forecast_raw.nc")
        gu.prepare_wind_file(DATA_DIR / "ecmwf_wind_forecast_raw.nc", DATA_DIR / "ecmwf_wind_gog_forecast.nc",
                             overwrite=True, bbox=(lon_min - 1, lon_max + 1, lat_min - 1, lat_max + 1))
    except ImportError:
        print("ecmwf-opendata is not installed: pip install ecmwf-opendata")
else:
    print("DOWNLOAD_DATA is False -> ECMWF open-data retrieval skipped")

### 7.6 Atmospheric forecast option C -- running your own WRF forecast (highest resolution, most effort)

For the same reason a nested CROCO run beats coarse GLORYS12/CMEMS forcing near the coast (Part 6), a
**regional WRF (Weather Research and Forecasting model)** run beats GFS/ECMWF's ~0.25 deg (~25 km) global
grid for GoG-specific effects: sea breezes, coastal convection, and orographic effects from Mount Cameroon --
all at scales of a few km, invisible in global atmospheric forecasts, but which can materially change the
wind-driven ("windage") component of oil, SAR and plastic drift right at the coast.

A minimal operational WRF forecast chain looks like:

1. Download GFS (Section 7.4) 0.25 deg forecast fields as initial/boundary conditions.
2. Run WRF preprocessing (`WPS`: `geogrid.exe`, `ungrib.exe`, `metgrid.exe`) to interpolate GFS onto your
   nested WRF domain (e.g. a 3 km Gulf of Guinea domain, mirroring the CROCO 1/36 deg footprint).
3. Run `real.exe` then `wrf.exe` for a 72-120 h forecast.
4. Post-process `wrfout_d0*` files to extract 10 m wind (`U10`, `V10`) on a regular lon/lat grid that
   `reader_netCDF_CF_generic` can ingest (WRF's native staggered Lambert/Mercator grid needs de-staggering
   and regridding first -- e.g. with `wrf-python`'s `getvar` + `xarray`'s `interp`).

This is beyond what we can run live in a summer-school session, but the WRF/WPS namelists and a worked
GFS-to-WRF-to-OpenDrift example are provided as a companion resource for participants with access to an
HPC/cloud account (e.g. via AGNES/ICMPA-UNESCO Chair computing resources).

In [ ]:
# Illustrative post-processing snippet, assuming a completed WRF run produced wrfout_d02_*.nc files
# already regridded to regular lon/lat with wrf-python (getvar(..., 'uvmet10') + horizontal interpolation)
if wrf_wind_file.exists():
    reader_wind_wrf = reader_netCDF_CF_generic.Reader(str(wrf_wind_file))
    print(reader_wind_wrf)
else:
    reader_wind_wrf = None
    print(f"No WRF wind file ({wrf_wind_file}): Part 7.7 will fall back to GFS, then ERA5 winds")

### 7.7 Putting it together: a fully operational, coastal-resolving forecast run

The following reproduces the **Part 2 oil-spill scenario**, but now entirely on operational forecast forcing:
CROCO 1/36 deg forecast currents + WRF 3 km forecast winds (falling back to GFS if WRF is unavailable,
and to ERA5 in this teaching setting if no forecast wind file is present).

In [ ]:
o_fc = OpenOil(loglevel=20)

wind_reader_to_use = next(r for r in [reader_wind_wrf, reader_wind_gfs, reader_wind] if r is not None)
print("Wind forcing used:", wind_reader_to_use.name)
o_fc.add_reader([reader_croco_fc, wind_reader_to_use])

o_fc.set_config('seed:oil_type', 'GENERIC MEDIUM CRUDE')
o_fc.set_config('processes:evaporation', True)
o_fc.set_config('processes:emulsification', True)
o_fc.set_config('processes:dispersion', True)
o_fc.set_config('drift:vertical_mixing', True)
o_fc.set_config('drift:current_uncertainty', 0.1)
o_fc.set_config('drift:wind_uncertainty', 1.0)

# Seed at the start of the forecast cycle ("now" for a real incident; the file's first time otherwise),
# and do not run beyond the end of the forecast.
if reader_croco_fc.start_time <= now <= reader_croco_fc.end_time - timedelta(hours=24):
    fc_seed_time = now
else:   # archived / placeholder cycle: start at the beginning of the forecast file
    fc_seed_time = reader_croco_fc.start_time
    print(f"'now' ({now}) is outside the forecast file: seeding at its first time, {fc_seed_time}")
fc_duration  = min(timedelta(hours=96), reader_croco_fc.end_time - fc_seed_time)
gu.seed_oil_mass(o_fc, oil_mass_kg, lon=release_lon, lat=release_lat, radius=100, number=2000,
                 time=fc_seed_time, z=0)
o_fc.run(duration=fc_duration, time_step=300, time_step_output=3600,
         outfile=str(OUT_DIR / "oil_operational_forecast.nc"))
o_fc.plot(fast=True, filename=str(OUT_DIR / "oil_operational_forecast_map.png"))

### 7.8 Forecast-specific caveats every operational user must know

- **Forecast skill degrades with lead time.** Always report drift predictions with an explicit *valid-until*
  time and re-run as soon as a fresher forecast cycle is available (every 6 h for GFS, daily for CROCO/CMEMS
  typically).
- **Boundary-condition lag.** A CROCO forecast's very first hours can show a short "spin-up" transient if its
  initial condition (from the global forecast) doesn't yet contain the CROCO-resolved mesoscale features --
  compare the first 6-12 h of a fresh forecast cycle against the end of the previous cycle's forecast as a
  sanity check.
- **Product availability changes.** Free forecast dataset IDs/endpoints (CMEMS `anfc` naming, NOMADS URL
  patterns, ECMWF open-data parameter lists) are updated by their providers from time to time -- always check
  the provider's current documentation before an incident, rather than trusting a hardcoded URL from an old
  script.
- **Ensemble/uncertainty.** Where available (e.g. ECMWF ENS, or running OpenDrift itself multiple times with
  `drift:current_uncertainty`/`drift:wind_uncertainty` perturbations), always prefer a **probability
  footprint** over a single deterministic track for incident-response products -- this is standard practice in
  the SAR and oil-spill response communities (cf. the ensemble philosophy of the ETOOFS Guide, IOC-UNESCO
  GOOS-275, 2022).

---

### EXERCISE 7.1 — Forecast versus reanalysis

Select an archived GFS forecast cycle for which forecast data are available (e.g. with
`gu.download_gfs_10m_wind(cycle, lead_hours, bbox, file)`, which also works for past cycles). Compare the GFS forecast
with ERA5 for the same valid time and geographical domain. Calculate and compare:

1. mean wind speed;
2. maximum wind speed;
3. mean wind-speed difference;
4. maximum absolute wind-speed difference;
5. wind-direction differences.

Plot the two wind fields and the corresponding difference field. Describe the main spatial patterns you observe,
then discuss possible reasons for the differences between the forecast and the reanalysis (forecast lead time,
resolution, assimilation of observations in the reanalysis but not or differently in the forecast).

### EXERCISE 7.2 — Global versus regional ocean forecasting

Run the same 72-hour oil-spill scenario using (a) a global ocean forecast (CMEMS, Section 7.2) and (b) the CROCO
regional forecast (Section 7.3). Compare the resulting simulations.

1. Compare the particle trajectories and spatial distributions, the time to first coastal impact, the fraction of
   particles reaching the coastline, and the spatial footprint of the oil.
2. Summarize the main differences you observe between the two simulations.
3. Discuss possible sources of uncertainty in the two forecast configurations (resolution, forecast lead time,
   boundary conditions, atmospheric forcing); try to distinguish sources of uncertainty that are specific to using
   a *forecast* (as opposed to a hindcast/reanalysis) from those that are about resolution alone.

### EXERCISE 7.3 — Global and regional ocean models in operational forecasting

Prepare a short briefing explaining why an operational marine forecasting system may use both global and regional
ocean models. Discuss: spatial resolution; computational cost; representation of coastal processes; initial and
boundary conditions; atmospheric forcing; availability of forecasts; observations and data assimilation; forecast uncertainty.

Conclude by explaining how global and regional models can complement each other in an operational
marine-forecasting system, and give one concrete example (from Parts 6–7 of this school) where you saw this
complementarity illustrated in your own results.

## Part 8 — References

- Auclair, F., Marchesiello, P., Debreu, L., et al. (2000 onward). *CROCO — Coastal and Regional Ocean
  COmmunity model.* https://www.croco-ocean.org
- Breivik, Ø., Allen, A.A., Maisondieu, C., Roth, J.C. (2011). Wind-induced drift of objects at sea: The
  leeway field method. *Applied Ocean Research*, 33(2), 100–109.
- Chelton, D.B., Schlax, M.G., Samelson, R.M. (2011). Global observations of nonlinear mesoscale eddies.
  *Progress in Oceanography*, 91(2), 167–216.
- Da-Allada, C.Y., et al. (2017). Causes for the recent increase in sea surface salinity in the Gulf of Guinea.
  *African Journal of Marine Science / Ocean Dynamics* (regional coastal-upwelling literature).
- Dagestad, K.-F., Röhrs, J., Breivik, Ø., Ådlandsvik, B. (2018). OpenDrift v1.0: a generic framework for
  trajectory modelling. *Geoscientific Model Development*, 11, 1405–1420.
- Debreu, L., Marchesiello, P., Penven, P., Cambon, G. (2012). Two-way nesting in split-explicit ocean models:
  algorithms, implementation and validation. *Ocean Modelling*, 49–50, 1–21.
- Djakouré, S., Penven, P., Bourlès, B., Veitch, J., Koné, V. (2017). Respective roles of the Guinea Current
  and local winds on the coastal upwelling in the northern Gulf of Guinea. *Journal of Physical Oceanography*,
  47(6), 1367–1387.
- Hersbach, H., et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological
  Society*, 146(730), 1999–2049.
- ICC International Maritime Bureau (annual). *Piracy and Armed Robbery Against Ships Report.*
- Jambeck, J.R., et al. (2015). Plastic waste inputs from land into the ocean. *Science*, 347(6223), 768–771.
- Johansen, Ø. (2000). DeepBlow — a Lagrangian plume model for deep water blowouts. *Spill Science &
  Technology Bulletin*, 6(2), 103–111.
- Lebreton, L.C.M., et al. (2017). River plastic emissions to the world's oceans. *Nature Communications*, 8,
  15611.
- Lellouche, J.-M., et al. (2021). The Copernicus Global 1/12° Oceanic and Sea Ice GLORYS12 Reanalysis.
  *Frontiers in Earth Science*, 9, 698876.
- Mason, E., Pascual, A., McWilliams, J.C. (2014). A new sea surface height–based code for oceanic mesoscale
  eddy tracking. *Journal of Atmospheric and Oceanic Technology*, 31(5), 1181–1188.
- Onink, V., et al. (2021). Global simulations of marine plastic transport show plastic trapping in coastal
  zones. *Environmental Research Letters*, 16, 064053.
- Röhrs, J., Dagestad, K.-F., Asbjørnsen, H., et al. (2018). The effect of vertical mixing on the horizontal
  drift of oil spills. *Ocean Science*, 14, 1581–1601.

### Data sources

- CMEMS GLORYS12V1 reanalysis: product `GLOBAL_MULTIYEAR_PHY_001_030`, Copernicus Marine Service,
  https://marine.copernicus.eu
- ERA5 hourly reanalysis: Copernicus Climate Data Store, https://cds.climate.copernicus.eu
- CROCO 1/36° Gulf of Guinea nested configuration: croco_work outputs.
- NOAA GFS: NOAA Open Data Dissemination (AWS `noaa-gfs-bdp-pds`), https://registry.opendata.aws/noaa-gfs-bdp-pds/

---

*Prepared for the GGOSSS Summer School 2026 — Gulf of Guinea francophone countries ocean summer school.
Please cite this notebook's software stack (OpenDrift, CROCO, CMEMS) in any derivative student work.*